# Cascade. End-to-end evaluation

**Protocol.**
- *Stage B* (headline): all 58,689 test images. An image blocked by the Stage I gate is scored
  as Grade 0, which is what the system outputs for it.
- *Stage A*: true Grade 1-5 images that reached Stage III (the grader in isolation).
- *Severity QWK*: images with true Grade 5 removed; a predicted Grade 5 on a gradable image is
  mapped to Grade 4. This is the definition used for the OOF figures.
- *Uncertainty*: 95% CIs from 1,000 image-level bootstrap resamples (500 for AUC), one shared
  generator seeded with 2026. The resamples are drawn in a fixed order, so the CIs reproduce
  exactly only when `run_final_evaluation` is run as a whole.

**Steps.**
1. Fit the thresholds on OOF, decode the test ensemble and compute every number.
2. Headline table (Stage A / B, tuned vs argmax, overlap-free subset).
3. QWK under the alternative definitions.
4. Referral view.
5. Where the cascade loses: error attribution, error distance, per-grade accuracy, confusion.
6. Per-dataset results.
7. Stage I operating point on the test pool.

## 0. Configuration

In [ ]:
from pathlib import Path

# Inputs (all per-image CSVs):
#   data/test/test_grade.csv                image, grade (0..5) for all 58,689 test images
#   data/test/stage1_manifest.csv           stem, prob, pred_binary (Stage I routing; 1 = passed)
#   outputs/stage3/eval/oof_corn.csv        OOF predictions (stage3_grading/02)
#   outputs/stage3/eval/lgdrg_test_predictions.csv   ensemble predictions (stage3_grading/03)
DATA_ROOT = Path("data")
GT_CSV = DATA_ROOT / "test" / "test_grade.csv"
MANIFEST_CSV = DATA_ROOT / "test" / "stage1_manifest.csv"

OUT_ROOT = Path("outputs")
OOF_CSV = OUT_ROOT / "stage3" / "eval" / "oof_corn.csv"
PRED_CSV = OUT_ROOT / "stage3" / "eval" / "lgdrg_test_predictions.csv"
CASCADE_DIR = OUT_ROOT / "cascade"
CASCADE_DIR.mkdir(parents=True, exist_ok=True)

GATE_THRESHOLD = 0.2391715943813324     # frozen Stage I operating point

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Compute all cascade numbers

Writes `final_numbers.json` and the tables behind every cascade result into `outputs/cascade/`.

In [ ]:
from hierarchiretina.stage3.cascade import run_final_evaluation

res = run_final_evaluation(OOF_CSV, PRED_CSV, GT_CSV, MANIFEST_CSV, CASCADE_DIR,
                           gate_threshold=GATE_THRESHOLD, verbose=True)
T = res["tables"]
C = T["cascade"]
print("thresholds:", res["calibration"]["t"], "t_g =", res["calibration"]["tg"])

## 2. Headline results

In [ ]:
h = res["headline"]
leak = T["leakage"].set_index("subset")
fmt = lambda v, ci=None: f"{v:.3f}" + (f" ({ci[0]:.3f}-{ci[1]:.3f})" if ci else "")
rows = [
    ["B, tuned", h["stageB_n"], fmt(h["stageB_qwk"], h["stageB_qwk_ci"]), h["stageB_acc"],
     h["stageB_macro_f1"]],
    ["B, argmax", h["stageB_n"], fmt(h["stageB_qwk_argmax"]), np.nan, np.nan],
    ["A, tuned", h["stageA_n"], fmt(h["stageA_qwk"], h["stageA_qwk_ci"]), h["stageA_acc"], np.nan],
    ["A, argmax", h["stageA_n"], fmt(h["stageA_qwk_argmax"]), np.nan, np.nan],
]
sub = leak.iloc[0]
rows += [["B, tuned (EyePACS+DDR+APTOS)", sub.n, fmt(sub.stageB_qwk), sub.stageB_acc, np.nan],
         ["A, tuned (EyePACS+DDR+APTOS)", sub.stageA_n, fmt(sub.stageA_qwk), np.nan, np.nan]]
pd.DataFrame(rows, columns=["view", "n", "QWK (95% CI)", "accuracy", "macro_F1"]).astype({"n": int}).round(3)

## 3. QWK definitions

`oof_match` is the reported definition. Keeping Grade 5 on the ordinal axis (`as_published`)
inflates kappa because ungradable images are easy to separate.

In [ ]:
from hierarchiretina.stage3.metrics import qwk_severity_modes

A = C[C.passed & (C.true_grade > 0)]
qa = qwk_severity_modes(A.true_grade, A.final, include_healthy=False)
qb = qwk_severity_modes(C.true_grade, C.final, include_healthy=True)
pd.DataFrame({"stage_A_qwk": {k: v[0] for k, v in qa.items()}, "stage_A_n": {k: v[1] for k, v in qa.items()},
              "stage_B_qwk": {k: v[0] for k, v in qb.items()}, "stage_B_n": {k: v[1] for k, v in qb.items()}}).round(4)

## 4. Referral view

Refer if the output is Grade 2 or higher, ungradable included. Vision-threatening DR: true
Grades 3-4 output as 3-4.

In [ ]:
r = res["referral"]
print(json.dumps(r, indent=1))
print(f"gate pass rate, true Grades 3-4: {res['gate']['sens_vision_threatening_3to4']:.3f}")

## 5. Where the cascade loses

In [ ]:
att, dist = res["attribution"], res["error_distance"]
tot = att["total"]
print("error attribution")
for k in ["stageI_missed_DR", "stageI_passed_healthy", "stageIII_wrong_on_DR"]:
    print(f"  {k:<24}{att[k]:>7,}  ({100 * att[k] / tot:.1f}%)")
print(f"  {'total':<24}{tot:>7,}")
err = {k: v for k, v in dist.items() if int(k) > 0}
n_err = sum(err.values())
print("\nStage III severity error distance (true and predicted in 1-4)")
for k, v in err.items():
    print(f"  |d| = {k}: {v:>6,}  ({100 * v / n_err:.1f}%)")

In [ ]:
T["stage3_test_per_grade"].round(3)

In [ ]:
cm = T["confusion"].values
cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(4.6, 4))
ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
for i in range(6):
    for j in range(6):
        ax.text(j, i, f"{cmn[i, j]:.2f}\n{cm[i, j]}", ha="center", va="center", fontsize=6.5,
                color="white" if cmn[i, j] > 0.5 else "black")
ax.set_xticks(range(6)); ax.set_yticks(range(6))
ax.set_xlabel("predicted grade"); ax.set_ylabel("true grade")
ax.set_title("End-to-end confusion (row-normalised)")
fig.tight_layout()

## 6. Per-dataset results

Source datasets are recognised from the image stem (`cascade.SOURCE_PATTERNS`). Gate
sensitivity and specificity are for Grade 0 versus 1-5.

In [ ]:
T["per_dataset"].round(3)

## 7. Stage I operating point on the test pool

In [ ]:
g = res["gate"]
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in g.items()})
T["gate_per_grade"].round(3)